# **Adversarial Search Strategies and Decision Trees**
### Unidade Curricular: Inteligência Artificial (CC2006)
### Ano Letivo 2025/2026

**Grupo 7 - PL6:** Gonçalo Sousa (202403669) / Guilherme Granjo (202404328) / Matilde Alves (202403407)
___

### **Índice**
1. Introdução
2. O Jogo PopOut
2.1. Regras Especiais
3. 
4. 
5. 
6. 


### **1. Introdução**
O presente projeto visa desenvolver e comparar duas abordagens de Inteligência Artificial, nomeadamente a implementação do algoritmo Monte Carlo Tree Search (MCTS) com Upper Confidence Bound for Trees (UCT) e árvores de decisão através do procedimento ID3 (árvore de decisão supervisionada), aplicadas a dois conjuntos de dados distintos.

O objetivo é compreender praticamente o funcionamento e especificações de cada abordagem estudada, em concreto:
- Implementar o jogo PopOut com interface que suporte os modos Humano vs. Humano, Humano vs. Computador e Computador vs. Computador;
- Implementar o algoritmo MCTS com UCT como método de avaliação;
- Explorar variantes do MCTS, tais como -----------;
- Implementar o algoritmo ID3 para construção de árvores de decisão;
- Aplicar o ID3 ao dataset Iris e ao dataset gerado pelo MCTS;
- Avaliar o desempenho das soluções desenvolvidas e discussão crítica dos resultados.

### **2. O Jogo PopOut**
O jogo PopOut é uma variante do jogo de estratégia Connect-4, com a possibilidade de adicionar e remover jogadas, conferindo ao jogo uma complexidade estratégica superior. Tal como no Connect-4, dois jogadores colocam alternadamente peças da sua cor no topo de qualquer uma das 7 colunas (*drop*).

A distinção fundamental do PopOut reside na existência de movimentos *pop*, em que um jogador porde remover uma das suas próprias peças da linha inferior do tabuleiro.

O objetivo dos jogos mantêm-se: ser o primeiro jogador a alinhar quatro peças da sua cor na horizontal, vertical ou diagonal.

#### **2.1. Regras Especiais**
O jogo PopOut define três regras adicionais para eliminar possíveis ambiguidades:
1. Vitória Simultânea: Se um movimento de *pop* criar 4-em-linha para ambos os jogadores, o jogador que executou o *pop* é o vencedor da partida;
2. Tabuleiro cheio: Se o tabuleiro estiver completamente preenchido, o jogador atual pode optar por realizar um *pop* ou declarar empate;
3. Regra da repetição: Se o mesmo estado do jogo se repetir 3 vezes, qualquer um dos jogadores pode declarar empate.


#### **2.3. Implementação do Jogo**
O jogo PopOut encontra-se implementado no ficheiro `popout.py`. Posteriormente será apresentado o ficheiro `game.py` que integrará todos os módulos desenvolvidos neste projeto.

Importações e constrains:


In [33]:
import numpy as np

ROWS, COLS = 6, 7
EMPTY, P1, P2 = 0, 1, 2
DIRECTIONS = [(0, 1), (1, 0), (1, 1), (1, -1)]
WINNER_DRAW = "draw"

A classe `Move` representa um movimento no jogo, nomeadamente uma coluna e o tipo de jogada: *drop* ou *pop*, com a respetiva validação de legibilidade.

In [34]:
class Move:
    def __init__(self, column, kind): #cria e inicializa o objeto
        if kind not in ("drop", "pop"):
            raise ValueError(f"invalid kind: {kind!r}")
        if not 0 <= column < COLS:
            raise ValueError(f"column out of [0,{COLS-1}]: {column}")
        self.column = column
        self.kind = kind

    def __eq__(self, other): #define a comparação com '=='
        if not isinstance(other, Move):
            return False
        return self.column == other.column and self.kind == other.kind

    def __hash__(self): #define o hash 
        return hash((self.column, self.kind))

    def __str__(self): #define a representação em string
        return f"{self.kind}({self.column})"

    def __repr__(self): #define a representação oficial

        return f"Move({self.column}, {self.kind!r})"

A classe `State` determina o estado imutável do jogo (estado base a cada nova jogada).

In [35]:
class State:
    def __init__(self, board, player_to_move, history_counts=None, last_move=None, winner=None):
        self.board = board # tabuleiro 6x7 com valores 0(vazio), 1(P1) e 2(P2)
        self.player_to_move = player_to_move # indicação do próximo jogador
        self.history_counts = history_counts if history_counts is not None else {} # deteção da tripla repetição de estados
        self.last_move = last_move # último movimento efetuado
        self.winner = winner # vencedor do jogo

A função `state_key` gera uma chave unica para um par (tabuleiro, jogador) que será usada para detetar as repetições.

A função `initial_state` cria e desenvolve o estado inicial do jogo, com tabuleiro vazio e jogador P1 a começar.

A função `legal_moves` devolve todos os movimentos que o jogador pode executar.

In [36]:
def state_key(board, player_to_move):
    return board.tobytes() + bytes([player_to_move])


def initial_state():
    board = np.zeros((ROWS, COLS), dtype=np.int8)
    history = {state_key(board, P1): 1}
    return State(board=board, player_to_move=P1, history_counts=history)


def legal_moves(state):
    if state.winner is not None:
        return []
    moves = []
    for c in range(COLS):
        if state.board[0, c] == EMPTY:
            moves.append(Move(c, "drop"))
        if state.board[ROWS - 1, c] == state.player_to_move:
            moves.append(Move(c, "pop"))
    return moves

As funções `_drop_row`, `_four_in_a_row_for`, `_has_legal_pop` e `_has_legal_drop` verificam vitórias e jogadas possíveis:

In [37]:
# Devolve a linha mais baixa vazia na coluna c ou -1 se a coluna estiver cheia
def _drop_row(board, c):
    for r in range(ROWS - 1, -1, -1):
        if board[r, c] == EMPTY:
            return r
    return -1

# Verifica se o jogador tem 4-em-linha
def _four_in_a_row_for(board, player):
    for r in range(ROWS):
        for c in range(COLS):
            if board[r, c] != player:
                continue
            for dr, dc in DIRECTIONS:
                rr, cc = r + 3 * dr, c + 3 * dc
                if 0 <= rr < ROWS and 0 <= cc < COLS:
                    if all(board[r + i * dr, c + i * dc] == player for i in range(4)):
                        return True
    return False

# Retorna True se for possível o jogador realizar um pop
def _has_legal_pop(board, player):
    return bool((board[ROWS - 1, :] == player).any())

# Retorna True se for possível o jogador realizar um drop
def _has_legal_drop(board):
    return bool((board[0, :] == EMPTY).any())

A função `check_win` verifica se existe um vencedor após um movimento, de acordo com as regras especiais do PopOut.

In [38]:
def check_win(board, last_move, mover):
    other = 3 - mover
    me_won = _four_in_a_row_for(board, mover)
    other_won = _four_in_a_row_for(board, other)

    if last_move.kind == "pop":
        if me_won:
            return mover
        if other_won:
            return other
    else:
        if me_won:
            return mover

    next_player = 3 - mover
    if not _has_legal_drop(board) and not _has_legal_pop(board, next_player):
        return WINNER_DRAW
    return None

A função `apply_move` aplica o ultimo movimento ao estado e devolve um novo estado imutável.

In [39]:
def apply_move(state, move):
    if state.winner is not None:
        raise ValueError("Game already finished.")

    new_board = state.board.copy()
    mover = state.player_to_move

    if move.kind == "drop":
        r = _drop_row(new_board, move.column)
        if r == -1:
            raise ValueError(f"Column {move.column} is full.")
        new_board[r, move.column] = mover
    else:
        if new_board[ROWS - 1, move.column] != mover:
            raise ValueError(f"Illegal pop on column {move.column}: bottom not player {mover}.")
        
        new_board[1:ROWS, move.column] = state.board[0:ROWS - 1, move.column]
        new_board[0, move.column] = EMPTY

    new_player = 3 - mover
    new_history = dict(state.history_counts)
    key = state_key(new_board, new_player)
    new_history[key] = new_history.get(key, 0) + 1

    winner = check_win(new_board, move, mover)

    return State(
        board=new_board,
        player_to_move=new_player,
        history_counts=new_history,
        last_move=move,
        winner=winner,
    )

As funções `can_claim_repetition_draw` e `render` determinam empate por repetição e reconfiguração do tabuleiro para o utilizador, respetivamente.

In [40]:
def can_claim_repetition_draw(state):
    key = state_key(state.board, state.player_to_move)
    return state.history_counts.get(key, 0) >= 3


def render(board):
    glyph = {EMPTY: "-", P1: "X", P2: "O"}
    lines = ["".join(glyph[int(v)] for v in row) for row in board]
    return "\n".join(lines)

### **3. Monte Carlo Tree Search (MCTS)**
O MCTS é um algoritmo de pesquisa heurística para tomada de decisões em grandes domínios de estados, como os jogos de tabuleiro, levando em consideração as constantes alterações causadas pelo adversário. A implementação utiliza o critério **Upper Confidence Bound for Trees (UCT)** para avaliar cada ramo da árvore, cuja fórmula é:

$$\text{UCB1}(n) = \underbrace{\frac{U(n)}{N(n)}}_{\text{exploitation}} + C \cdot \underbrace{\sqrt{\frac{\ln N(\text{parent}(n))}{N(n)}}}_{\text{exploration}}$$
<span style="font-size: 0.9em;">

> **Notação UCT (UCB1):** $n$ representa o nó a avaliar; $U(n)$ a utilidade acumulada; $N(n)$ o número de visitas ao nó $n$; $N_{parent(n)}$ o número de visitas ao nó pai de $n$; $C$ a constante de exploração.

</span>

A aplicação do algoritmo MCTS passa por 4 fases:
- **Selection:** A partir da raiz, escolhe o filho que maximiza o UCT até atingir um nó não totalmente expandido;
- **Expansion:** Adiciona um nó filho ao nó selecionado, representando um estado ainda não explorado;
- **Simulation:** A partir do novo nó, executa um jogo até ao fim, através de uma simulação aleatória ou heurística;
- **Backpropagation:** O resultado propaga-se de volta ao longo do percurso até à raíz, atualizando o contador de visitas em cada nó e o valor acumulado.

#### **3.1. Implementação do Algoritmo**
O algoritmo MCTS encontra-se implementado no ficheiro `mcts.py`.

Importações e constrains:

In [41]:
import math
import random

from popout import COLS, Move, State, apply_move, legal_moves

DEFAULT_C = math.sqrt(2)
DEFAULT_N_SIMULATIONS = 500
ROLLOUT_MAX_DEPTH = 200

CENTER = COLS // 2
COLUMN_PRIORITY = sorted(range(COLS), key=lambda c: abs(c - CENTER))

A função `_ordered_legal_moves` devolve as jogadas possíveis por proximidade ao centro.  

In [42]:
def _ordered_legal_moves(state, max_children):
    moves = legal_moves(state)
    if max_children is None or len(moves) <= max_children:
        return moves
    drops = [m for m in moves if m.kind == "drop"]
    pops = [m for m in moves if m.kind == "pop"]
    drops.sort(key=lambda m: abs(m.column - CENTER))
    pops.sort(key=lambda m: abs(m.column - CENTER))
    return (drops + pops)[:max_children]

A classe `Node` armazena o estado do nó, estatísticas UCB1 e filhos.

In [43]:
class Node:
    def __init__(self, state, parent=None, move_in=None, max_children=None): #Inicializa o nó da árvore MCTS
        self.state = state
        self.parent = parent
        self.move_in = move_in
        self.children = {}
        self.untried_moves = list(_ordered_legal_moves(state, max_children))
        self.N = 0
        self.U = 0.0

    def is_terminal(self): #Devolve True se o estado for terminal
        return self.state.winner is not None

    def is_fully_expanded(self): #Devolve True se todas as jogadas possíveis já foram exploradas
        return len(self.untried_moves) == 0 and len(self.children) > 0

    def best_child(self, c): #Seleciona o filho com maior valor UCB1
        log_N_parent = math.log(self.N) if self.N > 0 else 0.0

        def ucb1(child):
            if child.N == 0:
                return math.inf
            exploit = child.U / child.N
            explore = c * math.sqrt(log_N_parent / child.N)
            return exploit + explore

        return max(self.children.values(), key=ucb1)

    def expand(self, rng, max_children=None): #Expande o nó
        idx = rng.randrange(len(self.untried_moves))
        move = self.untried_moves.pop(idx)
        next_state = apply_move(self.state, move)
        child = Node(next_state, parent=self, move_in=move, max_children=max_children)
        self.children[move] = child
        return child

    def most_visited_child(self): #Devolve o filho com maior número de visitas
        return max(self.children.values(), key=lambda c: c.N)

A função `_find_winning_move` devolve a primeira jogada que resulta em vitória imediata.

A função `_move_is_safe` verifica se a jogada é segura, ou seja, se o adversário não vence imediatamente.

In [44]:
def _find_winning_move(state, moves):
    me = state.player_to_move
    for m in moves:
        ns = apply_move(state, m)
        if ns.winner == me:
            return m
    return None

def _move_is_safe(state, m):
    ns = apply_move(state, m)
    if ns.winner is not None:
        return True
    opp = ns.player_to_move
    for om in legal_moves(ns):
        nns = apply_move(ns, om)
        if nns.winner == opp:
            return False
    return True

A função `find_forced_win` procura uma jogada que garanta vitória independentemente da jogada do adversário.

In [45]:
def find_forced_win(state, depth=2):
    if depth < 1 or state.winner is not None:
        return None

    me = state.player_to_move
    moves = legal_moves(state)

    win_now = _find_winning_move(state, moves)
    if win_now is not None:
        return win_now
    if depth == 1:
        return None

    for m in moves:
        ns = apply_move(state, m)
        if ns.winner == me:
            return m
        if ns.winner is not None:
            continue
        opp_moves = legal_moves(ns)
        if not opp_moves:
            continue
        all_lead_to_my_win = True
        for om in opp_moves:
            nns = apply_move(ns, om)
            if nns.winner == me:
                continue
            if nns.winner is not None:
                all_lead_to_my_win = False
                break
            if _find_winning_move(nns, legal_moves(nns)) is None:
                all_lead_to_my_win = False
                break
        if all_lead_to_my_win:
            return m
    return None

A função `random_playout` faz uma simulação aleatória de uma partida até que alguém ganhe, empate ou atinja determinada profundidade.

A função `heuristic_win_playout` simula um jogo dando prioridade às vitorias imediatas, caso contrário, joga aleatoriamente.

A função `heuristic_block_playout` simula um jogo dando prioridade às vitórias imediatas e evita sempre a vitória do adversário.

In [46]:
def random_playout(state, rng, max_depth=ROLLOUT_MAX_DEPTH):
    depth = 0
    while state.winner is None and depth < max_depth:
        moves = legal_moves(state)
        if not moves:
            return "draw"
        idx = rng.randrange(len(moves))
        state = apply_move(state, moves[idx])
        depth += 1
    return state.winner if state.winner is not None else "draw"

def heuristic_win_playout(state, rng, max_depth=ROLLOUT_MAX_DEPTH):
    depth = 0
    while state.winner is None and depth < max_depth:
        moves = legal_moves(state)
        if not moves:
            return "draw"
        winning = _find_winning_move(state, moves)
        chosen = winning if winning is not None else moves[rng.randrange(len(moves))]
        state = apply_move(state, chosen)
        depth += 1
    return state.winner if state.winner is not None else "draw"

def heuristic_block_playout(state, rng, max_depth=ROLLOUT_MAX_DEPTH):
    depth = 0
    while state.winner is None and depth < max_depth:
        moves = legal_moves(state)
        if not moves:
            return "draw"
        winning = _find_winning_move(state, moves)
        if winning is not None:
            state = apply_move(state, winning)
            depth += 1
            continue
        safe = [m for m in moves if _move_is_safe(state, m)]
        pool = safe if safe else moves
        chosen = pool[rng.randrange(len(pool))]
        state = apply_move(state, chosen)
        depth += 1
    return state.winner if state.winner is not None else "draw"


ROLLOUTS = {
    "random": random_playout,
    "heuristic_win": heuristic_win_playout,
    "heuristic_block": heuristic_block_playout,
}

A função `backprop` propaga o resultado, após o rollout, para cima na árvore até à raíz.

In [47]:
def backprop(leaf, winner):
    node = leaf
    while node is not None:
        node.N += 1
        if node.parent is not None:
            chooser = node.parent.state.player_to_move
            if winner == chooser:
                node.U += 1.0
            elif winner == "draw":
                node.U += 0.5
        node = node.parent

A função `mcts_search` executa a implementação do algoritmo MCTS em si: realiza a pesquisa e devolve a melhor jogada a executar. 

A função `mcts_strategy` configura a função `mcts_search` e devolve uma função jogável.

In [48]:
def mcts_search(root_state, n_simulations=DEFAULT_N_SIMULATIONS, c=DEFAULT_C,
                rollout="random", max_children=None, tactical_root=False,
                tactical_depth=2, rng=None):

    if root_state.winner is not None:
        return None
    moves = legal_moves(root_state)
    if not moves:
        return None
    if len(moves) == 1:
        return moves[0]

    if tactical_root:
        forced = find_forced_win(root_state, depth=tactical_depth)
        if forced is not None:
            return forced

    if rollout not in ROLLOUTS:
        raise ValueError(f"invalid rollout: {rollout!r}. Choose from {list(ROLLOUTS)}.")
    playout_fn = ROLLOUTS[rollout]

    if rng is None:
        rng = random.Random()
    root = Node(root_state, max_children=max_children)

    for _ in range(n_simulations):
        node = root
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.best_child(c)
        if not node.is_terminal():
            node = node.expand(rng, max_children=max_children)
        winner = playout_fn(node.state, rng)
        backprop(node, winner)

    return root.most_visited_child().move_in


def mcts_strategy(n_simulations=DEFAULT_N_SIMULATIONS, c=DEFAULT_C, rollout="random",
                  max_children=None, tactical_root=False, tactical_depth=2, rng=None):

    if rng is None:
        rng = random.Random()

    def strat(state):
        move = mcts_search(
            state,
            n_simulations=n_simulations,
            c=c,
            rollout=rollout,
            max_children=max_children,
            tactical_root=tactical_root,
            tactical_depth=tactical_depth,
            rng=rng,
        )
        if move is None:
            raise RuntimeError("mcts_strategy called on terminal state.")
        return move

    return strat

#### **3.2. Outras estratégias de implementação**
As variações aplicadas ao algoritmo MCTS encontram-se implementadas no ficheiro `mcts_variations.py`.
As próximas funções e classe serão utilizadas nas novas abordagens apresentadas posteriormente.

Importações e constrains:

In [49]:
import argparse
import math
import random
import sys
import time

from game import play_game, random_strategy
from popout import P1, P2
from mcts import mcts_strategy

DEFAULT_C = math.sqrt(2)

A classe `MatchResult` armazena o resultado de jogo:

In [50]:
class MatchResult:
    def __init__(self, label_a, label_b, wins_a, wins_b, draws,
                 avg_time_a, avg_time_b, n_games):
        self.label_a = label_a
        self.label_b = label_b
        self.wins_a = wins_a
        self.wins_b = wins_b
        self.draws = draws
        self.avg_time_a = avg_time_a
        self.avg_time_b = avg_time_b
        self.n_games = n_games

A função `time_strategy` devolve o tempo gasto por jogada.

A função `run_match` executa partidas alterando o jogador que inicia a partida e devolve um `MatchResult` com as vitórias, empates e tempos médios por jogada.

In [51]:
def time_strategy(strat):
    times = []
    def wrapped(state):
        t0 = time.perf_counter()
        m = strat(state)
        times.append(time.perf_counter() - t0)
        return m
    wrapped._times = times
    return wrapped


def run_match(factory_a, factory_b, label_a, label_b,
              n_games=10, max_turns=300, seed_base=0):

    wins_a = 0
    wins_b = 0
    draws = 0
    times_a, times_b = [], []
    for g in range(n_games):
        if g % 2 == 0:
            sa = time_strategy(factory_a())
            sb = time_strategy(factory_b())
            p1, p2, a_player = sa, sb, P1
        else:
            sb = time_strategy(factory_b())
            sa = time_strategy(factory_a())
            p1, p2, a_player = sb, sa, P2
        final = play_game(p1, p2, on_render=lambda _: None,
                          show_intermediate=False, max_turns=max_turns)
        times_a.extend(sa._times)
        times_b.extend(sb._times)
        if final.winner == a_player:
            wins_a += 1
        elif final.winner == "draw":
            draws += 1
        else:
            wins_b += 1
    return MatchResult(
        label_a=label_a, label_b=label_b,
        wins_a=wins_a, wins_b=wins_b, draws=draws,
        avg_time_a=sum(times_a)/max(1, len(times_a)),
        avg_time_b=sum(times_b)/max(1, len(times_b)),
        n_games=n_games,
    )

As funções `factory_mcts` e `factory_random` retornam duas novas funções que criam, nomeadamente, uma estratégia MCTS e uma estratégia aleatória para inserção nos jogos.

In [52]:
def factory_mcts(n, c=DEFAULT_C, rollout="random", max_children=None, seed=None):
    def f():
        rng = random.Random(seed)
        return mcts_strategy(n_simulations=n, c=c, rollout=rollout,
                             max_children=max_children, rng=rng)
    return f

def factory_random(seed=None):
    def f():
        return random_strategy(random.Random(seed))
    return f

**Novas abordagens:**
- **A:** Compara rollouts *(random, heuristic_win, heuristic_block)*;
- **B:** Avalia o impacto do número de simulações *N* no desempenho do MCTS;
- **C:** Avalia o impacto da constante de exploração *c* no desempenho do MCTS;
- **D:** Avalia o impacto das limitações no *max_children* no desempenho do MCTS.

In [53]:
def experiment_A(quick):
    n_games = 6 if quick else 10
    N = 200
    print(f"\n=== A: rollout (N={N}, {n_games} games) ===\n")
    matchups = [
        ("random", "heuristic_win"),
        ("random", "heuristic_block"),
        ("heuristic_win", "heuristic_block"),
    ]
    results = []
    for ra, rb in matchups:
        print(f"  {ra} vs {rb} ...", flush=True)
        r = run_match(
            factory_mcts(N, rollout=ra, seed=0),
            factory_mcts(N, rollout=rb, seed=1),
            label_a=ra, label_b=rb, n_games=n_games,
        )
        print(f"    A {r.wins_a} - {r.wins_b} B (draws {r.draws}) "
              f"| t/move A={r.avg_time_a*1000:.0f}ms B={r.avg_time_b*1000:.0f}ms")
        results.append(r)
    return results


def experiment_B(quick):
    n_games = 4 if quick else 8
    print(f"\n=== B: N (rollout=heuristic_win, {n_games} games) ===\n")
    matchups = [(100, 300), (300, 600), (300, "random_baseline")]
    results = []
    for a, b in matchups:
        if b == "random_baseline":
            la, lb = f"MCTS_N{a}_heur", "random"
            print(f"  {la} vs {lb} ...", flush=True)
            r = run_match(
                factory_mcts(a, rollout="heuristic_win", seed=0),
                factory_random(seed=999),
                label_a=la, label_b=lb, n_games=n_games,
            )
        else:
            la, lb = f"MCTS_N{a}", f"MCTS_N{b}"
            print(f"  {la} vs {lb} ...", flush=True)
            r = run_match(
                factory_mcts(a, rollout="heuristic_win", seed=0),
                factory_mcts(b, rollout="heuristic_win", seed=1),
                label_a=la, label_b=lb, n_games=n_games,
            )
        print(f"    A {r.wins_a} - {r.wins_b} B (draws {r.draws}) "
              f"| t/move A={r.avg_time_a*1000:.0f}ms B={r.avg_time_b*1000:.0f}ms")
        results.append(r)
    return results


def experiment_C(quick):
    n_games = 6 if quick else 12
    N = 200
    print(f"\n=== C: C (N={N}, vs random, {n_games} games) ===\n")
    cs = [0.5, 1.0, math.sqrt(2), 2.0]
    results = []
    for c in cs:
        la = f"MCTS_C{c:.2f}"
        print(f"  {la} vs random ...", flush=True)
        r = run_match(
            factory_mcts(N, c=c, rollout="heuristic_win", seed=0),
            factory_random(seed=999),
            label_a=la, label_b="random", n_games=n_games,
        )
        print(f"    {la} {r.wins_a} - {r.wins_b} random (draws {r.draws}) "
              f"| t/move {r.avg_time_a*1000:.0f}ms")
        results.append(r)
    return results


def experiment_D(quick):
    n_games = 4 if quick else 8
    N = 200
    print(f"\n=== D: max_children (N={N}, vs random, {n_games} games) ===\n")
    ks = [None, 5, 3]
    results = []
    for k in ks:
        la = f"MCTS_k{k}"
        print(f"  {la} vs random ...", flush=True)
        r = run_match(
            factory_mcts(N, rollout="heuristic_win", max_children=k, seed=0),
            factory_random(seed=999),
            label_a=la, label_b="random", n_games=n_games,
        )
        print(f"    {la} {r.wins_a} - {r.wins_b} random (draws {r.draws}) "
              f"| t/move {r.avg_time_a*1000:.0f}ms")
        results.append(r)
    return results

#### **3.2.1. Execução das variações**


In [54]:
def print_table(title, results):
    print(f"\n## {title}\n")
    print(f"| {'A':<24} | {'B':<22} | A | B | D | t/A (ms) | t/B (ms) |")
    print(f"|{'-'*26}|{'-'*24}|---|---|---|----------|----------|")
    for r in results:
        print(f"| {r.label_a:<24} | {r.label_b:<22} | {r.wins_a} | {r.wins_b} | "
              f"{r.draws} | {r.avg_time_a*1000:>8.1f} | {r.avg_time_b*1000:>8.1f} |")


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--exp", choices=["A", "B", "C", "D"], default=None)
    if argv is None and "ipykernel" in sys.modules:
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args(argv)

    t0 = time.time()
    all_results = {}
    if args.exp in (None, "A"):
        all_results["A"] = experiment_A(args.quick)
    if args.exp in (None, "B"):
        all_results["B"] = experiment_B(args.quick)
    if args.exp in (None, "C"):
        all_results["C"] = experiment_C(args.quick)
    if args.exp in (None, "D"):
        all_results["D"] = experiment_D(args.quick)
    elapsed = time.time() - t0

    print(f"\n\n{'='*70}\nSUMMARY ({elapsed:.0f}s total)\n{'='*70}")
    if "A" in all_results:
        print_table("Experiment A -- rollout", all_results["A"])
    if "B" in all_results:
        print_table("Experiment B -- N", all_results["B"])
    if "C" in all_results:
        print_table("Experiment C -- C", all_results["C"])
    if "D" in all_results:
        print_table("Experiment D -- max_children", all_results["D"])
    return 0

# Para executar em notebook:
main()


=== A: rollout (N=200, 10 games) ===

  random vs heuristic_win ...


KeyboardInterrupt: 

Numa das execuções foi possível obter o seguinte output:

In [ ]:
=== A: rollout (N=200, 10 games) ===

  random vs heuristic_win ...
    A 0 - 10 B (draws 0) | t/move A=95ms B=239ms
  random vs heuristic_block ...
    A 10 - 0 B (draws 0) | t/move A=143ms B=10656ms
  heuristic_win vs heuristic_block ...
    A 0 - 10 B (draws 0) | t/move A=439ms B=13081ms

=== B: N (rollout=heuristic_win, 8 games) ===

  MCTS_N100 vs MCTS_N300 ...
    A 0 - 8 B (draws 0) | t/move A=179ms B=492ms
  MCTS_N300 vs MCTS_N600 ...
    A 0 - 8 B (draws 0) | t/move A=764ms B=1511ms
  MCTS_N300_heur vs random ...
    A 8 - 0 B (draws 0) | t/move A=461ms B=0ms

=== C: C (N=200, vs random, 12 games) ===

  MCTS_C0.50 vs random ...
    MCTS_C0.50 12 - 0 random (draws 0) | t/move 418ms
  MCTS_C1.00 vs random ...
    MCTS_C1.00 12 - 0 random (draws 0) | t/move 420ms
  MCTS_C1.41 vs random ...
    MCTS_C1.41 12 - 0 random (draws 0) | t/move 357ms
  MCTS_C2.00 vs random ...
    MCTS_C2.00 12 - 0 random (draws 0) | t/move 527ms

=== D: max_children (N=200, vs random, 8 games) ===

  MCTS_kNone vs random ...
    MCTS_kNone 8 - 0 random (draws 0) | t/move 335ms
  MCTS_k5 vs random ...
    MCTS_k5 8 - 0 random (draws 0) | t/move 267ms
  MCTS_k3 vs random ...
    MCTS_k3 8 - 0 random (draws 0) | t/move 221ms


======================================================================
SUMMARY (2248s total)
======================================================================

## Experiment A -- rollout

| A                        | B                      | A | B | D | t/A (ms) | t/B (ms) |
|--------------------------|------------------------|---|---|---|----------|----------|
| random                   | heuristic_win          | 0 | 10 | 0 |     95.5 |    239.3 |
| random                   | heuristic_block        | 10 | 0 | 0 |    143.3 |  10656.3 |
| heuristic_win            | heuristic_block        | 0 | 10 | 0 |    438.8 |  13081.1 |

## Experiment B -- N

| A                        | B                      | A | B | D | t/A (ms) | t/B (ms) |
|--------------------------|------------------------|---|---|---|----------|----------|
| MCTS_N100                | MCTS_N300              | 0 | 8 | 0 |    178.9 |    492.4 |
| MCTS_N300                | MCTS_N600              | 0 | 8 | 0 |    763.8 |   1510.9 |
| MCTS_N300_heur           | random                 | 8 | 0 | 0 |    460.9 |      0.0 |

## Experiment C -- C

| A                        | B                      | A | B | D | t/A (ms) | t/B (ms) |
|--------------------------|------------------------|---|---|---|----------|----------|
| MCTS_C0.50               | random                 | 12 | 0 | 0 |    418.4 |      0.0 |
| MCTS_C1.00               | random                 | 12 | 0 | 0 |    420.3 |      0.0 |
| MCTS_C1.41               | random                 | 12 | 0 | 0 |    357.0 |      0.0 |
| MCTS_C2.00               | random                 | 12 | 0 | 0 |    527.3 |      0.0 |

## Experiment D -- max_children

| A                        | B                      | A | B | D | t/A (ms) | t/B (ms) |
|--------------------------|------------------------|---|---|---|----------|----------|
| MCTS_kNone               | random                 | 8 | 0 | 0 |    335.1 |      0.0 |
| MCTS_k5                  | random                 | 8 | 0 | 0 |    266.8 |      0.0 |
| MCTS_k3                  | random                 | 8 | 0 | 0 |    221.3 |      0.0 |

**INTERPRETAÇÃO DE RESULTADOS**

#### **3.3. Criação do dataset**
Foi criado um dataset, com o algoritmo MCTS, para treinar uma árvore de decisão.
A geração encontra-se implementada no ficheiro `generate_dataaset.py` e produz o ficheiro `popout_dataset.csv`.

Importações necessárias:

In [ ]:
import argparse
import csv
import math
import os
import random
import sys
import time

from popout import COLS, Move, apply_move, initial_state, legal_moves
from mcts import mcts_strategy

As funções `encode_move`, `encode_state_row`, `feature_columns` e `write_header` codificam e estruturam os dados do jogo para serem inseridos no arquivo cvs.

In [ ]:
# Codificação de jogadas (ex.: 'd3' para drop - coluna 3, 'p0' para pop - coluna 0)
def encode_move(move):
    return f"{'d' if move.kind == 'drop' else 'p'}{move.column}"

# Codificação de estados numa lista de inteiros
def encode_state_row(state):
    flat = state.board.reshape(-1).tolist()
    return [int(v) for v in flat] + [int(state.player_to_move)]

# Devolve a lista dos nomes das colunas
def feature_columns():
    return [f"s{i}" for i in range(42)] + ["to_play"]

# Escreve a linha de cabeçalho (nomes das colunas) no CSV.
def write_header(writer):
    writer.writerow(feature_columns() + ["move"])


A função `generate_dataset` cria um dataset de pares (estado, movimento) a partir do jogo PopOut com o algoritmo MCTS.

In [56]:
def generate_dataset(n_games=50, out_path="popout_dataset.csv", epsilon=0.10, n_simulations=200, rollout="heuristic_win", tactical_root=True,
                     c=math.sqrt(2), seed=0, max_turns=300, verbose=True):
    
    if os.path.dirname(out_path):
        os.makedirs(os.path.dirname(out_path), exist_ok=True)

    rng_master = random.Random(seed)
    n_pairs = 0
    n_random_moves = 0
    n_mcts_moves = 0
    n_pops = 0
    game_lengths = []
    winners = {1: 0, 2: 0, "draw": 0, "incomplete": 0}
    move_class_counts = {}
    t0 = time.time()

    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        write_header(writer)

        for g in range(n_games):
            state = initial_state()
            game_seed = rng_master.randint(0, 10**9)
            rng_game = random.Random(game_seed)
            mcts = mcts_strategy(
                n_simulations=n_simulations, c=c,
                rollout=rollout, tactical_root=tactical_root,
                rng=random.Random(game_seed + 1),
            )

            turns = 0
            while state.winner is None and turns < max_turns:
                legal = legal_moves(state)
                if not legal:
                    break
                if rng_game.random() < epsilon:
                    move = legal[rng_game.randrange(len(legal))]
                    n_random_moves += 1
                else:
                    move = mcts(state)
                    n_mcts_moves += 1

                writer.writerow(encode_state_row(state) + [encode_move(move)])
                n_pairs += 1
                cls = encode_move(move)
                move_class_counts[cls] = move_class_counts.get(cls, 0) + 1
                if move.kind == "pop":
                    n_pops += 1

                state = apply_move(state, move)
                turns += 1

            game_lengths.append(turns)
            if state.winner in (1, 2):
                winners[state.winner] += 1
            elif state.winner == "draw":
                winners["draw"] += 1
            else:
                winners["incomplete"] += 1

            if verbose and (g + 1) % max(1, n_games // 10) == 0:
                elapsed = time.time() - t0
                rate = (g + 1) / elapsed
                eta = (n_games - g - 1) / rate if rate > 0 else 0
                print(f"  game {g+1}/{n_games}  pairs={n_pairs}  "
                      f"{elapsed:.0f}s elapsed  ETA {eta:.0f}s", flush=True)

    elapsed = time.time() - t0
    csv_size = os.path.getsize(out_path)
    return {
        "n_games": n_games, "n_pairs": n_pairs,
        "n_mcts_moves": n_mcts_moves, "n_random_moves": n_random_moves,
        "n_pops": n_pops, "pop_rate": n_pops / max(1, n_pairs),
        "epsilon": epsilon, "winners": winners,
        "avg_game_length": sum(game_lengths) / max(1, len(game_lengths)),
        "min_game_length": min(game_lengths) if game_lengths else 0,
        "max_game_length": max(game_lengths) if game_lengths else 0,
        "elapsed_s": elapsed, "csv_path": out_path, "csv_bytes": csv_size,
        "move_class_counts": dict(
            sorted(move_class_counts.items(), key=lambda kv: -kv[1])
        ),
    }

# Função main adaptada para possível execução no notebook
def main(out_path="popout_dataset.csv", n_games=50, epsilon=0.10, n_simulations=200, 
         rollout="heuristic_win", tactical=True, seed=0):
    print("=== Generate PopOut dataset ===")
    print(f"games={n_games} eps={epsilon} N={n_simulations} "
          f"rollout={rollout} tactical={tactical} seed={seed}\n")
    stats = generate_dataset(
        n_games=n_games, out_path=out_path, epsilon=epsilon,
        n_simulations=n_simulations, rollout=rollout,
        tactical_root=tactical, seed=seed,
    )
    print(f"\nPairs:        {stats['n_pairs']}  (mcts={stats['n_mcts_moves']} "
          f"random={stats['n_random_moves']})")
    print(f"Pops:         {stats['n_pops']} ({stats['pop_rate']*100:.1f}%)")
    print(f"Winners:      P1={stats['winners'][1]} P2={stats['winners'][2]} "
          f"draw={stats['winners']['draw']} incomplete={stats['winners']['incomplete']}")
    print(f"Game length:  avg {stats['avg_game_length']:.1f} plies "
          f"(min={stats['min_game_length']} max={stats['max_game_length']})")
    print(f"Time:         {stats['elapsed_s']:.1f}s")
    print(f"CSV:          {stats['csv_path']}  ({stats['csv_bytes']/1024:.1f} KB)")
    print("\nTop classes:")
    for cls, n in list(stats["move_class_counts"].items())[:8]:
        print(f"  {cls}: {n} ({n/stats['n_pairs']*100:.1f}%)")


# Execução da função com os mesmos parâmetros do código original
main(n_games=50, n_simulations=200)

=== Generate PopOut dataset ===
games=50 eps=0.1 N=200 rollout=heuristic_win tactical=True seed=0

  game 5/50  pairs=58  15s elapsed  ETA 132s
  game 10/50  pairs=158  38s elapsed  ETA 153s
  game 15/50  pairs=242  61s elapsed  ETA 143s
  game 20/50  pairs=329  87s elapsed  ETA 131s
  game 25/50  pairs=429  109s elapsed  ETA 109s
  game 30/50  pairs=504  127s elapsed  ETA 85s
  game 35/50  pairs=626  160s elapsed  ETA 69s
  game 40/50  pairs=684  176s elapsed  ETA 44s
  game 45/50  pairs=768  194s elapsed  ETA 22s
  game 50/50  pairs=856  219s elapsed  ETA 0s

Pairs:        856  (mcts=779 random=77)
Pops:         41 (4.8%)
Winners:      P1=33 P2=17 draw=0 incomplete=0
Game length:  avg 17.1 plies (min=7 max=34)
Time:         218.5s
CSV:          popout_dataset.csv  (75.4 KB)

Top classes:
  d3: 182 (21.3%)
  d4: 152 (17.8%)
  d2: 140 (16.4%)
  d5: 110 (12.9%)
  d1: 103 (12.0%)
  d6: 75 (8.8%)
  d0: 53 (6.2%)
  p3: 10 (1.2%)


#### **3.3. Árvores de Decisão (ID3)**

O algoritmo ID3 encontra-se implementado no ficheiro `decision_tree_builder.py`.

Importações necessárias:

In [ ]:
import math
import re
from collections import Counter

import numpy as np
import pandas as pd

A classe `Node` representa um nó da árvore de decisão ID3, que pode ser de dois tipos: nó interno (feature - atributo de separação // filhos) ou folha (label - classe prevista).

In [ ]:
class Node:
    def __init__(self, feature=None, children=None, label=None, n_samples=0, class_counts=None):
        self.feature = feature
        self.children = children if children is not None else {}
        self.label = label
        self.n_samples = n_samples
        self.class_counts = class_counts if class_counts is not None else {}

    @property
    def is_leaf(self):
        return self.label is not None

    def __repr__(self):
        if self.is_leaf:
            return f"Leaf(label={self.label!r}, n={self.n_samples})"
        return f"Node(feature={self.feature!r}, |children|={len(self.children)})"

A função `entropy` calcula a entropia da sequência de labels $y$, pela fórmula:

$$H(Y) = \sum_i - P({y_i}) \log_2 P({y_i})$$

In [ ]:
def entropy(y):
    if len(y) == 0:
        return 0.0
    counts = Counter(y)
    total = len(y)
    return -sum((c / total) * math.log2(c / total) for c in counts.values() if c)

A função `information_gain` calcula o ganho de informação de dividir X e Y pela feature especificada, pela fórmula:

$$\text{IG}(X) = H(Y) - \sum_j \frac{|S_j|}{|S|} H(Y \mid X=j)$$

In [ ]:
# X: DataFrame de atributos, y: Series de rótulos, feature: nome da coluna de X a avaliar

def information_gain(X, y, feature):
    base = entropy(y)
    total = len(y)
    if total == 0:
        return 0.0
    cond = 0.0
    for value in X[feature].unique():
        sub_y = y[X[feature] == value]
        if len(sub_y):
            cond += (len(sub_y) / total) * entropy(sub_y)
    return base - cond

A função principal `id3` cria recursivamente a árvore de decisão ID3 que retorna o nó raíz da árvore de decisão.

In [ ]:
def id3(X, y, features, parent_majority=None, max_depth=None, depth=0):
    counts = dict(Counter(y))
    n_samples = len(y)

    # Subconjunto vazio -> usa a classe maioritaria do pai
    if n_samples == 0:
        return Node(label=parent_majority, n_samples=0)

    # Subconjunto puro -> folha com essa classe
    if len(counts) == 1:
        only_class = next(iter(counts))
        return Node(label=only_class, n_samples=n_samples, class_counts=counts)

    # Sem features (ou limite de profundidade) -> folha pela maioria
    if not features or (max_depth is not None and depth >= max_depth):
        majority = max(counts.items(), key=lambda kv: kv[1])[0]
        return Node(label=majority, n_samples=n_samples, class_counts=counts)

    gains = [(f, information_gain(X, y, f)) for f in features]
    best_feature, best_gain = max(gains, key=lambda x: x[1])

    # Nenhum atributo separa nada: ruido/ambiguidade -> folha pela maioria
    if best_gain <= 0:
        majority = max(counts.items(), key=lambda kv: kv[1])[0]
        return Node(label=majority, n_samples=n_samples, class_counts=counts)

    node = Node(feature=best_feature, n_samples=n_samples, class_counts=counts)
    majority = max(counts.items(), key=lambda kv: kv[1])[0]
    remaining = [f for f in features if f != best_feature]

    for value in sorted(X[best_feature].unique(), key=str):
        mask = X[best_feature] == value
        node.children[value] = id3(
            X[mask], y[mask], remaining,
            parent_majority=majority,
            max_depth=max_depth,
            depth=depth + 1,
        )
    return node

As seguintes funções são responsáveis por inferir e avaliar resultados do uso da árvore, nomeadamente:
- `predict` tem como função prever a classe de um único exemplo, ou seja, percorre a árvore como um fluxo de decisões até obter a classe final;
- `predict_batch` prevê todos os exemplos do DataFrame e devolve uma lista de labels, de uma só vez;
- `accuracy` avalia a qualidade das previsões realizadas nas funções anteriores;
- `_majority_label` devolve a label com maior contagem total entre as folhas descendentes de um nó.

In [ ]:
def predict(tree, sample):
    node = tree
    while not node.is_leaf:
        value = sample.get(node.feature)
        if value in node.children:
            node = node.children[value]
        else:
            # Valor nunca visto neste no: vota entre as folhas descendentes
            return _majority_label(node)
    return node.label

def predict_batch(tree, X):
    return [predict(tree, row.to_dict()) for _, row in X.iterrows()]

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    if not y_true:
        return 0.0
    return sum(1 for a, b in zip(y_true, y_pred) if a == b) / len(y_true)

def _majority_label(node):
    if node.is_leaf:
        return node.label
    counts = Counter()
    stack = [node]
    while stack:
        n = stack.pop()
        if n.is_leaf:
            counts[n.label] += n.n_samples
        else:
            stack.extend(n.children.values())
    if not counts:
        return None
    return max(counts.items(), key=lambda kv: kv[1])[0]

Para que a árvore de decisão consiga fazer divisões mais interpretáveis, e uma vez que o algoritgmo ID3 funciona de forma mais estável com atributos categóricos, é necessária a **discretização** de valores numéricoss em categorias.

A classe `DiscretizationFit` armazena parâmetros do discretizador, tais como a estratégia de discretização *(por largura, frequência ou corte escolhido pelo ganho de informação)*, limites para cada coluna e o nome dos valores categóricos finais a serem usados pelo ID3.

As funções abaixo da classe permitem a distinção entre as diferentes estratégias a aplicar ao discretizador.

A última função `transform_discretizer` substitui os valores numéricos por labels.

In [ ]:
class DiscretizationFit:
    def __init__(self, strategy, edges, labels):
        self.strategy = strategy
        self.edges = edges
        self.labels = labels

# Ajusta o discretizador por largura igual
def fit_discretizer_equal_width(X, columns, n_bins=3):
    edges, labels = {}, {}
    for col in columns:
        col_min, col_max = float(X[col].min()), float(X[col].max())
        if col_min == col_max:
            edges[col] = np.array([col_min, col_max])
            labels[col] = ["bin0"]
            continue
        edges[col] = np.linspace(col_min, col_max, n_bins + 1)
        labels[col] = [f"bin{i}" for i in range(n_bins)]
    return DiscretizationFit("equal_width", edges, labels)

# Ajusta o discretizador por frequência igual
def fit_discretizer_equal_frequency(X, columns, n_bins=3):
    edges, labels = {}, {}
    for col in columns:
        quantiles = np.linspace(0, 1, n_bins + 1)
        e = np.unique(np.quantile(X[col].values, quantiles))
        edges[col] = e
        labels[col] = [f"bin{i}" for i in range(len(e) - 1)]
    return DiscretizationFit("equal_frequency", edges, labels)

# Ajusta o discretizador por ganho de informação
def fit_discretizer_supervised(X, y, columns):
    # Split binario por atributo no threshold que maximiza o IG
    edges, labels = {}, {}
    for col in columns:
        values = X[col].values
        sorted_vals = np.sort(np.unique(values))
        candidates = (sorted_vals[:-1] + sorted_vals[1:]) / 2
        base_h = entropy(y)
        n = len(y)
        best_t = None
        best_gain = -1.0
        for t in candidates:
            left, right = y[values <= t], y[values > t]
            if len(left) == 0 or len(right) == 0:
                continue
            cond = (len(left) / n) * entropy(left) + (len(right) / n) * entropy(right)
            gain = base_h - cond
            if gain > best_gain:
                best_gain = gain
                best_t = float(t)
        if best_t is None:
            best_t = float(np.median(values))
        edges[col] = np.array([-np.inf, best_t, np.inf])
        labels[col] = ["low", "high"]
    return DiscretizationFit("supervised", edges, labels)

def transform_discretizer(X, fit):
    out = X.copy()
    for col, e in fit.edges.items():
        if col not in out.columns:
            continue
        bins = np.searchsorted(e, out[col].values, side="right") - 1
        bins = np.clip(bins, 0, len(fit.labels[col]) - 1)
        out[col] = [fit.labels[col][i] for i in bins]
    return out


#### **3.3.1 Aplicação ao dataset `Iris`**

Para testar o algoritmo ID3, foi utilizado o dataset `iris.cvs` fornecido, de forma a treinar uma árvore decisão que, dadas quatro características (comprimento da sépala, largura da sépala, comprimento da pétala e largura da pétala), consiga determinar a que espécie cada planta pertence (*Iris setosa*, *Iris versicolor* e *Iris virginica*).